# Phase 3 — Physics-Constrained LLPS Prediction

Phase 1 established the generalization ladder on the LLPSense dataset. This notebook
builds the contribution that is ours rather than theirs.

**The idea.** At fixed everything else, raising protein concentration or crowder
percentage can only make phase separation *more* likely. That is not a statistical
assumption, it is phase-separation physics. A purely statistical model is free to
violate it, and LLPSense draws phase diagrams and ranks mutations from exactly such
a model, without ever checking.

XGBoost supports monotonic constraints, so the constraint can be imposed directly.
Salt, pH and temperature are deliberately left free: those genuinely are
non-monotonic (reentrant behaviour, LCST vs UCST).

**What this measures**
1. **Physical consistency** — a new metric: the fraction of predictions that fall as a
   constrained variable rises. Expected to be non-zero for the unconstrained model.
2. **Accuracy cost** — whether enforcing physics loses predictive performance.

On a preliminary run on the LLPSDB curation the answer was 5.0% violations, falling
to 0.0%, at no measurable accuracy cost. This notebook tests it on the LLPSense
feature basis.

**Setup:** Runtime -> T4 GPU, ~25-30 min.
Upload `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup and data

In [ ]:
import numpy as np, pandas as pd, subprocess, time, json, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
print("GPU:", GPU)

from google.colab import files
up = files.upload()        # llpsense_final_noseq.csv AND llpsense_t5.npz

df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
N_EMB = emb.shape[1]
print("features:", X.shape, "| embedding dims:", N_EMB)

GPU: True


Saving llpsense_final_noseq.csv to llpsense_final_noseq.csv
Saving llpsense_t5.npz to llpsense_t5.npz
features: (50557, 1037) | embedding dims: 1024


## 2. Define the physical constraints

Increasing-only: protein concentration and the six crowding agents.
Free: pH, temperature, MgCl2, NaCl, KCl (all genuinely non-monotonic), and glycerol,
whose effect is viscogenic as well as crowding and so is not a clean one-way variable.

In [ ]:
UP = ['Conc','PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']
cond_mono = [1 if c in UP else 0 for c in COND]
MONO = tuple([0]*N_EMB + cond_mono)       # embeddings unconstrained
print("constrained increasing:", UP)
print("left free:", [c for c in COND if c not in UP])
print("constraint vector length:", len(MONO), "= ", N_EMB, "+", len(COND))

constrained increasing: ['Conc', 'PEG300-1k', 'PEG3k-6k', 'PEG8k-20k', 'Ficoll', 'Dextran -40', 'Dextran 70-']
left free: ['pH', 'temp', 'MgCl2', 'NaCl', 'KCl', 'Glycerol']
constraint vector length: 1037 =  1024 + 13


## 3. The physical consistency metric

For a sample of real condition rows, sweep one constrained variable across its range
holding everything else fixed, and count how often the predicted probability *falls*
as that variable rises. A physically coherent model scores zero.

In [ ]:
def violation_rate(model, Xs, feat_idx, n_steps=12):
    """fraction of (row, step) pairs where p drops as the feature increases"""
    grid = np.linspace(0.0, 1.0, n_steps)      # conditions are min-max normalised
    prev, viol, tot = None, 0, 0
    for v in grid:
        Xt = Xs.copy(); Xt[:, feat_idx] = v
        p = model.predict_proba(Xt)[:, 1]
        if prev is not None:
            viol += int((p < prev - 1e-6).sum()); tot += len(p)
        prev = p
    return viol / tot

IDX = {c: N_EMB + i for i, c in enumerate(COND)}
rng = np.random.default_rng(0)
SAMPLE = rng.choice(len(X), size=min(2000, len(X)), replace=False)
print("consistency probe rows:", len(SAMPLE))

consistency probe rows: 2000


## 4. Train both models across three regimes

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, brier_score_loss
from xgboost import XGBClassifier

HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899,
          random_state=42)
RUNTIME = dict(tree_method="hist", eval_metric="logloss",
               **({"device": "cuda"} if GPU else {"n_jobs": -1}))

REGIMES = [("A  new condition, known protein", "RecordID"),
           ("B  new protein",                  "SeqID"),
           ("C* new family (hardened)",        "GroupX")]
VARIANTS = [("unconstrained (LLPSense-style)", {}),
            ("physics-constrained (ours)",     {"monotone_constraints": MONO})]

res = {}
for vname, vkw in VARIANTS:
    for rname, key in REGIMES:
        g, rows, t0 = df[key].to_numpy(), [], time.time()
        for i, j in StratifiedGroupKFold(5, shuffle=True, random_state=1).split(X, y, groups=g):
            m = XGBClassifier(**HP, **vkw, **RUNTIME)
            m.fit(X[i], y[i], sample_weight=w[i])
            p = m.predict_proba(X[j])[:, 1]
            if len(np.unique(y[j])) < 2: continue
            rows.append([roc_auc_score(y[j], p, sample_weight=w[j]),
                         average_precision_score(y[j], p, sample_weight=w[j]),
                         accuracy_score(y[j], p > .5, sample_weight=w[j]),
                         brier_score_loss(y[j], p, sample_weight=w[j])])
        a = np.array(rows); res[(vname, rname)] = a
        print(f"{vname:32s} {rname:34s} ROC-AUC {a[:,0].mean():.3f} ±{a[:,0].std():.3f}"
              f"  ({time.time()-t0:.0f}s)", flush=True)
json.dump({f"{k[0]}|{k[1]}": v.tolist() for k, v in res.items()}, open("phase3_regimes.json","w"))

unconstrained (LLPSense-style)   A  new condition, known protein    ROC-AUC 0.834 ±0.009  (137s)
unconstrained (LLPSense-style)   B  new protein                     ROC-AUC 0.783 ±0.013  (141s)
unconstrained (LLPSense-style)   C* new family (hardened)           ROC-AUC 0.648 ±0.040  (142s)
physics-constrained (ours)       A  new condition, known protein    ROC-AUC 0.833 ±0.008  (150s)
physics-constrained (ours)       B  new protein                     ROC-AUC 0.776 ±0.011  (148s)
physics-constrained (ours)       C* new family (hardened)           ROC-AUC 0.641 ±0.029  (146s)


## 5. Physical consistency

Both variants retrained on all data, then probed. This is the headline comparison.

In [ ]:
cons = {}
for vname, vkw in VARIANTS:
    m = XGBClassifier(**HP, **vkw, **RUNTIME)
    m.fit(X, y, sample_weight=w)
    row = {}
    for c in ['Conc', 'PEG3k-6k', 'Ficoll', 'NaCl']:     # NaCl is a free-variable control
        row[c] = violation_rate(m, X[SAMPLE], IDX[c])
    cons[vname] = row
    print(f"{vname:32s} " + "  ".join(f"{k} {100*v:5.1f}%" for k, v in row.items()), flush=True)

print("\nNaCl is unconstrained in both models — it should show violations in both,")
print("confirming the metric detects real non-monotonicity rather than always reading zero.")
json.dump(cons, open("phase3_consistency.json","w"))

unconstrained (LLPSense-style)   Conc   6.2%  PEG3k-6k   1.0%  Ficoll   0.2%  NaCl  29.3%
physics-constrained (ours)       Conc   0.0%  PEG3k-6k   0.0%  Ficoll   0.0%  NaCl  26.3%

NaCl is unconstrained in both models — it should show violations in both,
confirming the metric detects real non-monotonicity rather than always reading zero.


## 6. Results table

In [ ]:
rows = []
for (vname, rname), a in res.items():
    rows.append({"model": vname, "regime": rname,
                 "ROC-AUC": f"{a[:,0].mean():.3f} ± {a[:,0].std():.3f}",
                 "PR-AUC": f"{a[:,1].mean():.3f}", "Accuracy": f"{a[:,2].mean():.3f}",
                 "Brier": f"{a[:,3].mean():.3f}"})
tab = pd.DataFrame(rows).pivot(index="regime", columns="model")
print(tab.to_string())

print("\n=== accuracy cost of enforcing physics ===")
for rname, _ in REGIMES:
    u = res[("unconstrained (LLPSense-style)", rname)][:,0]
    c = res[("physics-constrained (ours)", rname)][:,0]
    print(f"  {rname:34s} {u.mean():.3f} -> {c.mean():.3f}   ({c.mean()-u.mean():+.3f})")

print("\n=== physical consistency (violation rate) ===")
for v, r in cons.items():
    print(f"  {v:32s} concentration {100*r['Conc']:.1f}%   PEG {100*r['PEG3k-6k']:.1f}%")

pd.DataFrame(rows).to_csv("phase3_results.csv", index=False)
from google.colab import files
files.download("phase3_results.csv"); files.download("phase3_consistency.json")

                                                   ROC-AUC                                                    PR-AUC                                                  Accuracy                                                     Brier                               
model                           physics-constrained (ours) unconstrained (LLPSense-style) physics-constrained (ours) unconstrained (LLPSense-style) physics-constrained (ours) unconstrained (LLPSense-style) physics-constrained (ours) unconstrained (LLPSense-style)
regime                                                                                                                                                                                                                                                                 
A  new condition, known protein              0.833 ± 0.008                  0.834 ± 0.009                      0.845                          0.848                      0.763                          0.760   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### What to look for

- **Violations non-zero for the unconstrained model** — this is the finding. It means
  LLPSense-style models can and do predict that adding protein makes demixing less
  likely, which cannot happen physically.
- **Violations zero for the constrained model** — guaranteed by construction, but worth
  confirming the metric reports it.
- **ROC-AUC difference near zero** — physics is free. If instead accuracy drops
  noticeably, that is also publishable: it would mean the model was buying accuracy
  from physically impossible regions of the condition space, which is worse.
- **NaCl showing violations in both** — a sanity check that the metric works.